# Dicke Synchronization with a Collective Spin Ensemble

This notebook uses the symmetric Dicke manifold to simulate a driven, collectively damped spin ensemble. The Hilbert-space dimension is only `N + 1`, so it is a compact way to explore finite-size collective dynamics before moving to larger trajectory studies.

If you are running this notebook in Colab, install the package first:

```python
%pip install openquantumsim
```

The first solver call on a new machine may spend a few minutes resolving and precompiling the Julia backend.

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np

import openquantumsim as oqs

In [ ]:
def spin_coherent_dicke(space: oqs.DickeSpace, theta: float, phi: float = 0.0) -> np.ndarray:
    """Spin-coherent state in the Dicke basis ordered by descending excitations."""
    n_spins = space.n_spins
    ket = np.zeros(space.dim, dtype=np.complex128)
    for excitations in range(n_spins + 1):
        amplitude = (
            math.sqrt(math.comb(n_spins, excitations))
            * np.sin(theta / 2) ** excitations
            * np.cos(theta / 2) ** (n_spins - excitations)
            * np.exp(1j * phi * excitations)
        )
        ket[n_spins - excitations] = amplitude
    return ket / np.linalg.norm(ket)

In [ ]:
n_spins = 24
space = oqs.DickeSpace(n_spins, label="ensemble")

omega = 1.0
gamma = 0.35
times = np.linspace(0.0, 12.0, 241)

psi0 = spin_coherent_dicke(space, theta=np.pi / 3, phi=0.0)
rho0 = oqs.ket2dm(psi0)

H = omega * oqs.collective_x(space)
c_ops = [np.sqrt(gamma / n_spins) * oqs.collective_lowering(space)]

jx = (2.0 / n_spins) * oqs.collective_x(space)
jz = (2.0 / n_spins) * oqs.collective_z(space)
excitation = (1.0 / n_spins) * oqs.collective_excitation(space)

In [ ]:
result = oqs.mesolve(
    H,
    rho0,
    times,
    c_ops=c_ops,
    e_ops=[jx, jz, excitation],
    options=oqs.Options(rtol=1e-8, atol=1e-10),
)

In [ ]:
fig, ax = plt.subplots(figsize=(7.0, 4.0))
ax.plot(times, result.expect[0].real, label=r"$2\langle J_x\rangle/N$")
ax.plot(times, result.expect[1].real, label=r"$2\langle J_z\rangle/N$")
ax.plot(times, result.expect[2].real, label=r"$\langle n_e\rangle/N$")
ax.set_xlabel("time")
ax.set_ylabel("normalized expectation")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()

Try changing `n_spins`, `omega`, and `gamma`. For trajectory studies, replace `mesolve` with `mcsolve` and keep the same Dicke-space operators.